In [2]:
import os
import operator
from typing import TypedDict, Annotated, List, Union
from dotenv import load_dotenv

# LangGraph and LangChain imports
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_google_community import GoogleSearchAPIWrapper 
from langchain_core.exceptions import OutputParserException

c:\Users\TPWODL\miniconda3\envs\genai\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [5]:
# --- 1. Define the Agent State (The Shared Whiteboard) ---
# This defines the schema for the state passed between nodes in the graph.
class AgentState(TypedDict):
    """
    Represents the state of the agent's execution.
    The 'messages' key is handled by a special reducer (add_messages) to append new messages.
    """
    messages: Annotated[List[BaseMessage], add_messages]
    # Keep track of the number of tool execution attempts (optional, but good for safety)
    tool_calls_count: int

# --- 2. Define the External Tools ---

# NOTE: You must set the following environment variables:
# 1. OPENAI_API_KEY
# 2. GOOGLE_API_KEY (from Google Cloud Console)
# 3. GOOGLE_CSE_ID (from Google Programmable Search Engine)
# Load environment variables from a .env file if it exists
load_dotenv()

# The Google Search API Wrapper from LangChain
search = GoogleSearchAPIWrapper()

# Define the Google Search tool that the LLM can call
google_search_tool = Tool(
    name="google_search",
    description="Search Google for current information or specific facts on a topic. Returns text snippets, titles, and URLs.",
    func=lambda q: search.results(q, num_results=5)
)

TOOLS = [google_search_tool]

# --- 3. Agent and Node Definitions ---

# Initialize the LLM (OpenAI) with tool-calling capabilities
# Using gpt-4o-mini for speed and cost-effectiveness in this ReAct loop
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)


ValidationError: 1 validation error for GoogleSearchAPIWrapper
  Value error, Did not find google_api_key, please add an environment variable `GOOGLE_API_KEY` which contains it, or pass `google_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error